# MCP SCENARIO: “Smart IT Helpdesk Assistant”
 Scenario Background

 You are working in a company called ABC Corp.

 Employees face issues like:

 VPN not working
 Printer not responding
 Software errors

 👉 Instead of calling IT support, employees use an AI Helpdesk Bot.

 🤖 What this Bot Should Do

 When a user types a problem:

 Understand the issue
 Decide if a ticket is needed
 Identify:
 Category (Network / Hardware / General)
 Priority (High / Medium)
 Create a ticket
 Show confirmation
 🧠 How MCP Fits Here
 Component	Role in Scenario
 Agent	Helpdesk Bot
 MCP Layer	Decision + Tool calling
 Tool	Ticket Creation System
 User	Employee

In [ ]:
# ============================================
# STEP 0: DATABASE (Simulated storage)
# ============================================

tickets_db = []  # This stores all tickets


# ============================================
# STEP 1: TOOL (MCP TOOL)
# ============================================

def create_ticket(issue, priority, category):
    """
    This function simulates a TOOL in MCP
    In real world → API / Database / ServiceNow
    """

    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket


# ============================================
# STEP 2: AGENT REASONING (LLM SIMULATION)
# ============================================

def analyze_input(user_input):
    """
    Simulates how an LLM understands user input
    Extracts:
    - category
    - priority
    """

    text = user_input.lower()

    # 🔹 Category Detection
    if "vpn" in text:
        category = "network"
    elif "printer" in text:
        category = "hardware"
    elif "email" in text:
        category = "software"
    else:
        category = "general"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text:
        priority = "high"
    elif "slow" in text:
        priority = "low"
    else:
        priority = "medium"

    return category, priority


# ============================================
# STEP 3: DECISION ENGINE (MCP CORE)
# ============================================

def should_call_tool(user_input):
    """
    Decides whether to call a tool or not
    This is MCP decision layer
    """

    keywords = ["issue", "problem", "ticket", "not working"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 4: MCP ORCHESTRATOR
# ============================================

def mcp_agent(user_input):
    """
    This is the MAIN MCP FLOW
    It connects:
    Agent → Decision → Tool → Response
    """

    print("\n🧠 Agent received input:", user_input)

    # STEP 4.1: Decision
    if should_call_tool(user_input):

        print("➡️ Decision: Tool call required")

        # STEP 4.2: Analyze input
        category, priority = analyze_input(user_input)

        print(f"📊 Extracted → Category: {category}, Priority: {priority}")

        # STEP 4.3: Prepare payload (MCP format)
        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("📦 MCP Payload:", payload)

        # STEP 4.4: Call tool
        result = create_ticket(**payload)

        print("⚙️ Tool executed successfully")

        # STEP 4.5: Final response
        return f"""
        ✅ Ticket Created Successfully!

        Ticket ID: {result['ticket_id']}
        Issue: {result['issue']}
        Category: {result['category']}
        Priority: {result['priority']}
        """

    else:
        print("➡️ Decision: No tool needed (AI response)")

        return "🤖 AI Response: Please describe your issue clearly."


# ============================================
# STEP 5: RUN INTERACTIVE LOOP
# ============================================

print("🚀 MCP Demo Started (Type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting MCP demo...")
        break

    response = mcp_agent(user_input)
    print(response)


🚀 MCP Demo Started (Type 'exit' to stop)

Enter your query: Netwrok Issue urgent

🧠 Agent received input: Netwrok Issue urgent
➡️ Decision: Tool call required
📊 Extracted → Category: general, Priority: high
📦 MCP Payload: {'issue': 'Netwrok Issue urgent', 'priority': 'high', 'category': 'general'}
⚙️ Tool executed successfully

        ✅ Ticket Created Successfully!

        Ticket ID: INC1000
        Issue: Netwrok Issue urgent
        Category: general
        Priority: high
        
Enter your query: exit
👋 Exiting MCP demo...


In [ ]:
!pip install groq
import os
from groq import Groq
from google.colab import userdata
from openai import OpenAI

# Load API key securely
api_key = userdata.get("Apii")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"  # This tells OpenAI to talk to Groq
)

# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []

# ============================================
# STEP 1: TOOL
# ============================================

def create_ticket(issue, priority, category):
    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: LLM ANALYSIS (REPLACES RULES)
# ============================================

def analyze_with_llm(user_input):
    """
    LLM decides:
    - should_create_ticket
    - category
    - priority
    """

    prompt = f"""
You are an IT helpdesk assistant.

Analyze the user issue and respond in JSON format:

{{
  "create_ticket": true/false,
  "category": "network/hardware/software/general",
  "priority": "high/medium/low"
}}

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # fast + powerful
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium"
        }

    return parsed


# ============================================
# STEP 3: MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 Agent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
✅ Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
"""

    else:
        return "🤖 AI Response: No ticket required. Try basic troubleshooting."


# ============================================
# STEP 4: RUN LOOP
# ============================================

print("🚀 LLM MCP Helpdesk Started (type 'exit')\n")

while True:

    user_input = input("Enter issue: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

🚀 LLM MCP Helpdesk Started (type 'exit')

Enter issue: Network Issue urgent

🧠 Agent received: Network Issue urgent
🤖 LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'Network Issue urgent', 'priority': 'medium', 'category': 'general'}

✅ Ticket Created Successfully!

Ticket ID: INC1000
Issue: Network Issue urgent
Category: general
Priority: medium

Enter issue: CPU Blast Urgent

🧠 Agent received: CPU Blast Urgent
🤖 LLM Decision: {'create_ticket': True, 'category': 'hardware', 'priority': 'high'}
📦 MCP Payload: {'issue': 'CPU Blast Urgent', 'priority': 'high', 'category': 'hardware'}

✅ Ticket Created Successfully!

Ticket ID: INC1001
Issue: CPU Blast Urgent
Category: hardware
Priority: high

Enter issue: exit
👋 Exiting...


#MCP SCENARIO: “Smart HR Onboarding Assistant”
🧩 Scenario Background
You are working in a company called XYZ Corp.
New employees often face challenges during onboarding, such as:
- Trouble accessing payroll portal
- Confusion about leave policies
- Difficulty setting up email accounts
- Questions about training schedules
👉 Instead of emailing HR or waiting for responses, employees use an AI Onboarding Bot.

🤖 What this Bot Should Do
When a new hire types a question/problem:
- Understand the query (e.g., “I can’t log into payroll”)
- Decide if escalation to HR is needed
- Identify:
- Category (Payroll / Policy / IT Setup / Training)
- Priority (High / Medium)
- Create a support ticket if required
- Provide instant guidance (FAQs, step-by-step instructions)
- Show confirmation and next steps

🧠 How MCP Fits Here
|  |  |
|  |  |
|  |  |
|  |  |
|  |  |



This way, the MCP framework is reused in a Human Resources context, where the AI assistant streamlines onboarding, reduces HR workload, and ensures employees feel supported from day one.
Would you like me to design another variation in a customer service setting (like retail or banking), so you can compare how MCP adapts across industries?

In [ ]:
# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []

# ============================================
# STEP 1: TOOL
# ============================================

def create_ticket(issue, priority, category):
    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: LLM ANALYSIS
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are an HR onboarding assistant.

Analyze the user query and respond in JSON:

{{
  "create_ticket": true/false,
  "category": "payroll/policy/it_setup/training/general",
  "priority": "high/medium/low",
  "response": "short helpful answer for employee"
}}

User Query: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium",
            "response": "We are looking into your issue."
        }

    return parsed


# ============================================
# STEP 3: MCP AGENT
# ============================================

def hr_mcp_agent(user_input):

    print("\n🧠 Agent received:", user_input)

    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    # Always show instant guidance
    reply = f"💡 Help: {decision['response']}\n"

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        reply += f"""
✅ HR Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

📩 HR team will contact you shortly.
"""

    else:
        reply += "✅ No ticket needed. Try the above steps."

    return reply


# ============================================
# STEP 4: RUN LOOP
# ============================================

print("🚀 HR Onboarding Assistant Started (type 'exit')\n")

while True:
    user_input = input("Ask HR Bot: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = hr_mcp_agent(user_input)
    print(response)

🚀 HR Onboarding Assistant Started (type 'exit')

Ask HR Bot: what is notice period policy

🧠 Agent received: what is notice period policy
🤖 LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium', 'response': 'We are looking into your issue.'}
📦 MCP Payload: {'issue': 'what is notice period policy', 'priority': 'medium', 'category': 'general'}
💡 Help: We are looking into your issue.

✅ HR Ticket Created!

Ticket ID: HR1000
Category: general
Priority: medium

📩 HR team will contact you shortly.

Ask HR Bot: exit
👋 Exiting...


# With Api


In [ ]:
# ============================================
# INSTALL (Colab only)
# ============================================
!pip install groq

# ============================================
# IMPORTS
# ============================================

import os
import json
from groq import Groq
from google.colab import userdata
from openai import OpenAI

# ============================================
# API KEY
# ============================================

api_key = userdata.get("Apii")  # Make sure key is saved in Colab

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

# ============================================
# DATABASE
# ============================================

tickets_db = []

# ============================================
# TOOL: CREATE TICKET
# ============================================

def create_ticket(issue, priority, category):
    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket

# ============================================
# LLM ANALYSIS (SMART HR BRAIN)
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are a Smart HR Onboarding Assistant.

Your job:
- Understand employee problem
- Decide if ticket is needed
- Classify category
- Assign priority
- Provide helpful response

Return ONLY JSON:

{{
  "create_ticket": true/false,
  "category": "payroll/policy/it_setup/training/general",
  "priority": "high/medium/low",
  "response": "short helpful answer for user"
}}

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium",
            "response": "We are checking your issue. HR will contact you."
        }

    return parsed

# ============================================
# MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 Agent received:", user_input)

    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    # Always show help response
    reply = f"\n💡 Help: {decision['response']}\n"

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        reply += f"""
✅ HR Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

📩 HR team will contact you soon.
"""

    else:
        reply += "✅ No ticket required."

    return reply

# ============================================
# RUN LOOP
# ============================================

print("🚀 Smart HR Onboarding Assistant Started (type 'exit')\n")

while True:

    user_input = input("Ask HR Bot: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

🚀 Smart HR Onboarding Assistant Started (type 'exit')

Ask HR Bot: what is the notice period policy 

🧠 Agent received: what is the notice period policy 
🤖 LLM Decision: {'create_ticket': False, 'category': 'policy', 'priority': 'low', 'response': "Our company's notice period policy can be found in the employee handbook, which states that employees are required to provide a minimum of 2 weeks' notice prior to leaving the company. If you have any specific questions or concerns, please let me know and I'll be happy to help."}

💡 Help: Our company's notice period policy can be found in the employee handbook, which states that employees are required to provide a minimum of 2 weeks' notice prior to leaving the company. If you have any specific questions or concerns, please let me know and I'll be happy to help.
✅ No ticket required.
Ask HR Bot: exit'

🧠 Agent received: exit'
🤖 LLM Decision: {'create_ticket': False, 'category': 'general', 'priority': 'low', 'response': 'Goodbye. If you need 

#MCP SCENARIO: “Smart Banking Support Assistant”
🧩 Scenario Background
You are working in a company called FinTrust Bank.
Customers often face issues such as:
- Credit card not working
- Trouble with online banking login
- Queries about loan status
- Transaction disputes
👉 Instead of calling customer care, customers use an AI Banking Support Bot.

🤖 What this Bot Should Do
When a customer types a problem:
- Understand the issue (e.g., “My card was declined”)
- Decide if escalation to a human agent is needed
- Identify:
- Category (Card Services / Online Banking / Loans / Transactions)
- Priority (High / Medium)
- Create a support ticket if required
- Provide instant guidance (FAQs, troubleshooting steps, policy info)
- Show confirmation and next steps

🧠 How MCP Fits Here
|  |  |
|  |  |
|  |  |
|  |  |
|  |  |



This way, MCP is applied in a financial services context, where the AI assistant reduces call center load, provides quick resolutions, and ensures customers feel supported with secure, reliable guidance.
Would you like me to craft one more in a healthcare setting (like hospital patient support), so you can see how MCP adapts to critical service environments?

In [ ]:
# STEP 0 & 1: Database and Tool
banking_tickets = []

def create_banking_ticket(issue, category, priority):
    t_id = f"BANK-{5000 + len(banking_tickets)}"
    ticket = {"id": t_id, "issue": issue, "category": category, "priority": priority}
    banking_tickets.append(ticket)
    return ticket

# STEP 2: Logic (Hard-coded Rules)
def analyze_banking_rules(user_input):
    ui = user_input.lower()
    if "card" in ui or "declined" in ui:
        return {"create": True, "cat": "Card Services", "prio": "High"}
    elif "login" in ui or "password" in ui:
        return {"create": True, "cat": "Online Banking", "prio": "Medium"}
    elif "loan" in ui or "mortgage" in ui:
        return {"create": True, "cat": "Loans", "prio": "Low"}
    elif "charge" in ui or "dispute" in ui:
        return {"create": True, "cat": "Transactions", "prio": "High"}
    return {"create": False}

# STEP 3 & 4: Agent and Run Loop
print("🏦 FinTrust Bank Support (Rule-Based) - Type 'exit' to quit\n")
while True:
    user_input = input("How can we help with your account? ").strip()
    if user_input.lower() == "exit": break
    if not user_input: continue

    decision = analyze_banking_rules(user_input)
    if decision["create"]:
        res = create_banking_ticket(user_input, decision["cat"], decision["prio"])
        print("\n--- 🎫 BANKING TICKET GENERATED ---")
        print(f"ID      : {res['id']}\nCategory: {res['category']}\nPriority: {res['priority']}\nIssue   : {res['issue']}")
        print("----------------------------------\n")
    else:
        print("🤖 AI: Please visit our branch or check our FAQ online.\n")

🏦 FinTrust Bank Support (Rule-Based) - Type 'exit' to quit

How can we help with your account? credit card issue

--- 🎫 BANKING TICKET GENERATED ---
ID      : BANK-5000
Category: Card Services
Priority: High
Issue   : credit card issue
----------------------------------

How can we help with your account? exit


with llm

In [ ]:
import os, json
from openai import OpenAI
from dotenv import load_dotenv
from google.colab import userdata
api_key=userdata.get('Apii')

load_dotenv()
client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")

banking_tickets = []

def create_banking_ticket(issue, category, priority):
    t_id = f"BANK-{5000 + len(banking_tickets)}"
    ticket = {"id": t_id, "issue": issue, "category": category, "priority": priority}
    banking_tickets.append(ticket)
    return ticket

def analyze_banking_llm(user_input):
    prompt = f"""You are a Banking Assistant. Categorize this issue: "{user_input}"
    Return JSON ONLY: {{"create": true, "category": "Card Services/Online Banking/Loans/Transactions", "priority": "High/Medium/Low"}}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

print("🚀 FinTrust AI Support (LLM-Powered) - Type 'exit' to quit\n")
while True:
    user_input = input("Describe your banking issue: ").strip()
    if user_input.lower() == "exit": break
    if not user_input: continue

    decision = analyze_banking_llm(user_input)

    if decision["create"]:
        res = create_banking_ticket(user_input, decision["category"], decision["priority"])
        print("\n--- 🏦 SECURE TICKET CREATED ---")
        print(f"Ticket ID : {res['id']}")
        print(f"Category  : {res['category']}")
        print(f"Priority  : {res['priority']}")
        print(f"Reference : {res['issue']}")
        print("--------------------------------\n")
    else:
        print("🤖 AI: No ticket required. Most login issues can be solved by resetting your password.\n")

🚀 FinTrust AI Support (LLM-Powered) - Type 'exit' to quit

Describe your banking issue: credit card issue

--- 🏦 SECURE TICKET CREATED ---
Ticket ID : BANK-5000
Category  : Card Services
Priority  : Medium
Reference : credit card issue
--------------------------------

Describe your banking issue: exi
🤖 AI: No ticket required. Most login issues can be solved by resetting your password.

Describe your banking issue: exit


#Create a  Weather Tool MCP Server that any AI agent can use with sample use case

In [ ]:
import requests
import json

# ============================================
# THE MCP SERVER (Weather Service)
# ============================================

class WeatherMCPServer:
    def __init__(self):
        self.name = "WeatherService"
        self.base_url = "https://api.open-meteo.com/v1/forecast"

    def get_tools(self):
        """Returns the definitions of tools available on this server."""
        return [{
            "name": "get_weather",
            "description": "Get current temperature and weather for a location using coordinates.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {"type": "number"},
                    "longitude": {"type": "number"}
                },
                "required": ["latitude", "longitude"]
            }
        }]

    def call_tool(self, tool_name, args):
        """Executes the logic for a specific tool."""
        if tool_name == "get_weather":
            params = {
                "latitude": args["latitude"],
                "longitude": args["longitude"],
                "current_weather": True
            }
            response = requests.get(self.base_url, params=params)
            data = response.json()

            if "current_weather" in data:
                weather = data["current_weather"]
                return {
                    "temp": f"{weather['temperature']}°C",
                    "wind": f"{weather['windspeed']} km/h",
                    "condition_code": weather['weathercode']
                }
        return {"error": "Tool not found"}

# Initialize Server
weather_server = WeatherMCPServer()

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from google.colab import userdata
# from openai import OpenAI

# ============================================
# API KEY
# ============================================

api_key = userdata.get("Apii")  # Make sure key is saved in Colab

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

def weather_mcp_agent(user_prompt):
    print(f"\n🔍 Agent thinking about: '{user_prompt}'")

    # 1. Ask LLM if it needs a tool
    # Added a 'system' message - Groq functions better with this
    messages = [
        {"role": "system", "content": "You are a helpful assistant with access to a weather tool. Use the tool if the user asks about weather or coordinates."},
        {"role": "user", "content": user_prompt}
    ]

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=[{"type": "function", "function": weather_server.get_tools()[0]}],
            tool_choice="auto",
            temperature=0  # Keeping temp at 0 helps with tool call stability
        )

        msg = response.choices[0].message

        # 2. Check if the LLM wants to call the Tool
        if msg.tool_calls:
            tool_call = msg.tool_calls[0]
            # Groq sometimes returns a string that needs careful parsing
            args = json.loads(tool_call.function.arguments)

            print(f"🛰️ AI calling Server Tool: {tool_call.function.name} with {args}")

            # 3. Execute the tool on the Server
            tool_result = weather_server.call_tool(tool_call.function.name, args)

            return f"🤖 AI Response: The current temperature is {tool_result['temp']} with winds at {tool_result['wind']}."

        return f"🤖 AI Response: {msg.content}"

    except Exception as e:
        return f"❌ Error during tool call: {e}"
# ============================================
# SAMPLE USE CASE: TRAVEL PLANNING
# ============================================
print("🌍 Weather MCP Agent Active (type 'exit')")

while True:
    query = input("\nAsk about weather (e.g., 'What is the weather in London at 51.5, -0.12?'): ")
    if query.lower() == 'exit': break

    result = weather_mcp_agent(query)
    print(result)

🌍 Weather MCP Agent Active (type 'exit')

Ask about weather (e.g., 'What is the weather in London at 51.5, -0.12?'): what is the Wheather in Noida

🔍 Agent thinking about: 'what is the Wheather in Noida'
🛰️ AI calling Server Tool: get_weather with {'latitude': 28.5743, 'longitude': 77.3232}
🤖 AI Response: The current temperature is 29.3°C with winds at 6.5 km/h.

Ask about weather (e.g., 'What is the weather in London at 51.5, -0.12?'): exit
